In [ ]:
import gc
import torch
import shutil
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from ultralytics import YOLO
from PIL import Image
from tqdm import tqdm
import warnings
import os

In [ ]:
def get_image_size(img_path):
    with Image.open(img_path) as img:
        return img.size

def yolo_bbox(row, img_w, img_h, n_classes):
    class_id = int(row['class_id'])
    if class_id >= n_classes:
        return None

    x_min = max(0, min(row['x_min'], img_w))
    y_min = max(0, min(row['y_min'], img_h))
    x_max = max(0, min(row['x_max'], img_w))
    y_max = max(0, min(row['y_max'], img_h))

    if x_max <= x_min or y_max <= y_min:
        return None

    x_center = max(0, min((x_min + x_max) / 2 / img_w, 1))
    y_center = max(0, min((y_min + y_max) / 2 / img_h, 1))
    width = max(0, min((x_max - x_min) / img_w, 1))
    height = max(0, min((y_max - y_min) / img_h, 1))

    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"

def generate_yolo_labels(df, image_ids, img_dir, labels_dir, img_sizes_cache, n_classes):
    os.makedirs(labels_dir, exist_ok=True)
    for img_id in image_ids:
        img_path = os.path.join(img_dir, f"{img_id}.jpg")
        if img_id not in img_sizes_cache:
            img_sizes_cache[img_id] = get_image_size(img_path)
        img_w, img_h = img_sizes_cache[img_id]

        mask = df['image_id'] == img_id
        txt_path = os.path.join(labels_dir, f"{img_id}.txt")
        with open(txt_path, 'w') as f:
            if not mask.any():
                continue
            for _, row in df[mask].iterrows():
                line = yolo_bbox(row, img_w, img_h, n_classes)
                if line is not None:
                    f.write(line + '\n')

def create_yaml(train_paths, val_paths, nc=14, names=None):
    if names is None:
        names = [f"class_{i}" for i in range(nc)]
    yaml_content = f"""
                        path: .
                        train: {train_paths}
                        val: {val_paths}
                        nc: {nc}
                        names: {names}
                    """
    yaml_path = "temp_dataset.yaml"
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    return yaml_path


def save_prediction_to_disk(image_id, preds_list, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"{image_id}.pkl")
    with open(file_path, 'wb') as f:
        pickle.dump(preds_list, f)


def run_cross_validation_yolo(df_raw, img_dir, n_splits=5, epochs=20, n_classes=14,
                              imgsz=640, batch=8, project='cv_fold', 
                              predictions_dir='model_predictions'):
    """
    Make cross-validation and save preds on disk
    """
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"
    
    df_clean = df_raw[df_raw['class_id'] < n_classes].copy()
    if len(df_clean) < len(df_raw):
        print(f"🗑️  Удалено {len(df_raw)-len(df_clean)} строк с фоновыми классами")

    image_ids = df_clean['image_id'].unique()
    print(f"The total number of unique images: {len(image_ids)}")

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    img_sizes_cache = {}

    os.makedirs(predictions_dir, exist_ok=True)

    for fold, (train_idx, val_idx) in enumerate(kf.split(image_ids)):
        train_ids = image_ids[train_idx]
        val_ids = image_ids[val_idx]
        print(f"\n🔁 Fold {fold+1}/{n_splits}: Train={len(train_ids)}, Val={len(val_ids)}")

        train_labels_dir = f"{project}/fold_{fold}/labels/train"
        val_labels_dir = f"{project}/fold_{fold}/labels/val"
        generate_yolo_labels(df_clean, train_ids, img_dir, train_labels_dir, img_sizes_cache, n_classes)
        generate_yolo_labels(df_clean, val_ids, img_dir, val_labels_dir, img_sizes_cache, n_classes)

        train_img_dir = os.path.abspath(f"{project}/fold_{fold}/images/train")
        val_img_dir = os.path.abspath(f"{project}/fold_{fold}/images/val")
        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(val_img_dir, exist_ok=True)

        for img_id in train_ids:
            src = os.path.join(img_dir, f"{img_id}.jpg")
            dst = os.path.join(train_img_dir, f"{img_id}.jpg")
            if not os.path.exists(dst) and os.path.exists(src):
                os.symlink(src, dst)
        for img_id in val_ids:
            src = os.path.join(img_dir, f"{img_id}.jpg")
            dst = os.path.join(val_img_dir, f"{img_id}.jpg")
            if not os.path.exists(dst) and os.path.exists(src):
                os.symlink(src, dst)

        yaml_path = create_yaml(
            os.path.abspath(train_img_dir),
            os.path.abspath(val_img_dir),
            nc=n_classes
        )

        model = YOLO('yolov8m.pt')
        model.train(
            data=yaml_path,
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            device=0,
            project=project,
            name=f'fold_{fold}',
            exist_ok=True,
            verbose=False
        )

        model = YOLO(f"{project}/fold_{fold}/weights/best.pt")
        results = model.predict(
            source=val_img_dir,  
            imgsz=imgsz,
            conf=0.1,
            iou=0.5,
            stream=True,
            verbose=False
        )

        print(f"Start predictions for valid images...")

        # --- Обрабатываем результаты по одному ---
        for res in tqdm(results, desc=f"   Saving fold {fold+1}"):
            # Из пути к файлу получаем image_id
            file_path = res.path
            image_id = os.path.splitext(os.path.basename(file_path))[0]
            # Проверяем, что image_id есть в val_ids (на всякий случай)
            if image_id not in val_ids:
                # Если не совпадает, пробуем fallback
                print(f"   ⚠️ {image_id} не в val_ids, пропускаем")
                continue

            boxes = res.boxes
            if boxes is None or len(boxes) == 0:
                save_prediction_to_disk(image_id, [], predictions_dir)
                continue

            xyxy = boxes.xyxy.cpu().numpy()
            conf = boxes.conf.cpu().numpy()
            cls = boxes.cls.cpu().numpy().astype(int)

            preds_list = []
            for i in range(len(xyxy)):
                class_id = cls[i]
                if class_id >= n_classes:
                    continue
                bbox = xyxy[i].tolist()
                conf_score = conf[i]
                probs = np.full(n_classes, (1 - conf_score) / (n_classes - 1), dtype=np.float32)
                probs[class_id] = conf_score
                preds_list.append({
                    'bbox': bbox,
                    'class': int(class_id),
                    'confidence': float(conf_score),
                    'class_probs': probs
                })
            save_prediction_to_disk(image_id, preds_list, predictions_dir)

        del model
        torch.cuda.empty_cache()
        gc.collect()

        try:
            shutil.rmtree(f"{project}/fold_{fold}")
        except Exception as e:
            print(f"Cannot delete fold_{fold}: {e}")

    print(f"\nAll predictions are saved in {predictions_dir}")
    return None

def load_predictions(predictions_dir='model_predictions'):
    all_preds = {}
    for fname in os.listdir(predictions_dir):
        if fname.endswith('.pkl'):
            image_id = fname.replace('.pkl', '')
            with open(os.path.join(predictions_dir, fname), 'rb') as f:
                all_preds[image_id] = pickle.load(f)
    return all_preds

def load_prediction(image_id, predictions_dir='model_predictions'):
    file_path = os.path.join(predictions_dir, f"{image_id}.pkl")
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            return pickle.load(f)
    return None

In [ ]:
df = pd.read_csv("data/VinDr-CXR/images/baseline_labels.csv")

# 2. Путь к изображениям
img_dir = "data/VinDr-CXR/images/train"
project = "data/cv_fold"
predictions_dir = "data/VinDr-CXR/model_predictions_10/model_predictions"

In [ ]:
run_cross_validation_yolo(
    df_raw=df,
    img_dir=img_dir,
    project=project,
    predictions_dir=predictions_dir,
    n_splits=5,
    epochs=10,
    n_classes=14,
    imgsz=640,
    batch=16
)